# Gold Layer: Traffic Congestion Forecast

This notebook analyzes traffic hazard patterns and forecasts congestion to help EV drivers:
- **Identify high-traffic periods** - When roads are most congested
- **Recommend charging windows** - Best times to charge before travel
- **Route risk assessment** - Areas with frequent traffic incidents
- **Seasonal patterns** - Weekly and daily congestion trends

## Business Questions Answered:

### 1. **When is the worst time to drive?**
Analyze hazards by hour and day of week to identify peak congestion periods

### 2. **Where are the congestion hotspots?**
Identify LGAs and regions with the highest incident density

### 3. **What types of hazards are most common?**
Classify hazards (roadwork, incidents, floods) and their temporal patterns

### 4. **When should EV drivers charge?**
Recommend off-peak hours for charging to avoid rush hour travel

## Data Sources:
- **Silver Layer**: `mobility_ai.silver.traffic_hazards`
- **Silver Layer**: `mobility_ai.silver.ev_charging_stations`

## Gold Tables Created:
- `gold.congestion_forecast_hourly` - Hourly congestion patterns
- `gold.congestion_forecast_location` - Location-based risk scores
- `gold.congestion_forecast_recommendations` - Optimal charging time windows

In [0]:
# Configuration
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timedelta
import math

# Table paths
TRAFFIC_HAZARDS_TABLE = "mobility_ai.silver.traffic_hazards"
EV_STATIONS_TABLE = "mobility_ai.silver.ev_charging_stations"

# Output tables
HOURLY_FORECAST_TABLE = "mobility_ai.gold.congestion_forecast_hourly"
LOCATION_FORECAST_TABLE = "mobility_ai.gold.congestion_forecast_location"
RECOMMENDATIONS_TABLE = "mobility_ai.gold.congestion_forecast_recommendations"

# Analysis parameters
HIGH_CONGESTION_THRESHOLD = 20  # Number of hazards to consider high congestion
PEAK_HOURS = [7, 8, 9, 17, 18, 19]  # Rush hours
OFF_PEAK_HOURS = [0, 1, 2, 3, 4, 5, 22, 23]  # Low traffic hours

print("\n" + "═" * 80)
print("⚙️  CONGESTION FORECAST ENGINE - CONFIGURATION")
print("═" * 80)
print(f"\n📊 Source Tables:")
print(f"   • Traffic Hazards:    {TRAFFIC_HAZARDS_TABLE}")
print(f"   • EV Stations:        {EV_STATIONS_TABLE}")
print(f"\n🎯 Output Tables:")
print(f"   • Hourly Forecast:    {HOURLY_FORECAST_TABLE}")
print(f"   • Location Forecast:  {LOCATION_FORECAST_TABLE}")
print(f"   • Recommendations:    {RECOMMENDATIONS_TABLE}")
print(f"\n⚡ Parameters:")
print(f"   • High Congestion:    {HIGH_CONGESTION_THRESHOLD}+ hazards/hour")
print(f"   • Peak Hours:         {PEAK_HOURS}")
print(f"   • Off-Peak Hours:     {OFF_PEAK_HOURS}")
print("═" * 80 + "\n")

In [0]:
# Load traffic hazards and EV stations
traffic_hazards = spark.table(TRAFFIC_HAZARDS_TABLE)
ev_stations = spark.table(EV_STATIONS_TABLE)

hazard_count = traffic_hazards.count()
station_count = ev_stations.count()
active_hazards = traffic_hazards.filter(F.col("is_active") == True).count()

print("\n" + "═" * 80)
print("📊 DATA LOADING")
print("═" * 80)
print(f"\n✅ Source Tables Loaded:")
print(f"   • Total Traffic Hazards:   {hazard_count:,}")
print(f"   • Active Hazards:          {active_hazards:,} ({active_hazards/hazard_count*100:.1f}%)")
print(f"   • EV Charging Stations:    {station_count:,}")
print("\n" + "─" * 80)
print("\n📋 Traffic Hazards Schema:")
print(f"   • Key fields: start_time_ts, end_time_ts, hazard_type_clean, severity")
print(f"   • Location: latitude, longitude, lganame")
print(f"   • Status: is_active, is_major")
print("═" * 80 + "\n")

In [0]:
# Extract temporal features from hazards
hourly_analysis = traffic_hazards.filter(
    F.col("start_time_ts").isNotNull()
).withColumn(
    "hour_of_day", F.hour(F.col("start_time_ts"))
).withColumn(
    "day_of_week", F.dayofweek(F.col("start_time_ts"))  # 1=Sunday, 7=Saturday
).withColumn(
    "day_name",
    F.when(F.col("day_of_week") == 1, "Sunday")
     .when(F.col("day_of_week") == 2, "Monday")
     .when(F.col("day_of_week") == 3, "Tuesday")
     .when(F.col("day_of_week") == 4, "Wednesday")
     .when(F.col("day_of_week") == 5, "Thursday")
     .when(F.col("day_of_week") == 6, "Friday")
     .when(F.col("day_of_week") == 7, "Saturday")
     .otherwise("Unknown")
)

# Aggregate by hour and day of week
hourly_forecast = hourly_analysis.groupBy(
    "hour_of_day",
    "day_of_week",
    "day_name"
).agg(
    # Hazard counts
    F.count("*").alias("total_hazards"),
    F.sum(F.when(F.col("severity") == "Major", 1).otherwise(0)).alias("major_hazards"),
    F.sum(F.when(F.col("severity") == "Moderate", 1).otherwise(0)).alias("moderate_hazards"),
    F.sum(F.when(F.col("severity") == "Minor", 1).otherwise(0)).alias("minor_hazards"),
    
    # Hazard type breakdown
    F.sum(F.when(F.col("main_category") == "Scheduled Roadwork", 1).otherwise(0)).alias("roadwork_count"),
    F.sum(F.when(F.col("main_category") == "Hazard", 1).otherwise(0)).alias("incident_count"),
    F.sum(F.when(F.col("main_category") == "Flooding", 1).otherwise(0)).alias("flood_count"),
    F.sum(F.when(F.col("main_category") == "Special Event", 1).otherwise(0)).alias("event_count"),
    
    # Active hazards
    F.sum(F.when(F.col("is_active"), 1).otherwise(0)).alias("active_hazards")
)

# Calculate congestion score (0-100)
hourly_forecast = hourly_forecast.withColumn(
    "congestion_score",
    F.least(
        F.lit(100),
        (
            # Base score from total hazards (max 50 points)
            F.least(F.lit(50), F.col("total_hazards") * 2) +
            
            # Major hazard penalty (max 30 points)
            (F.col("major_hazards") * 10) +
            
            # Active hazard factor (max 20 points)
            F.least(F.lit(20), F.col("active_hazards"))
        )
    ).cast("int")
)

# Categorize congestion level
hourly_forecast = hourly_forecast.withColumn(
    "congestion_level",
    F.when(F.col("congestion_score") >= 70, "Critical")
     .when(F.col("congestion_score") >= 50, "High")
     .when(F.col("congestion_score") >= 30, "Moderate")
     .when(F.col("congestion_score") >= 10, "Low")
     .otherwise("Minimal")
)

# Add peak hour indicator
hourly_forecast = hourly_forecast.withColumn(
    "is_peak_hour",
    F.col("hour_of_day").isin(PEAK_HOURS)
)

hourly_forecast = hourly_forecast.withColumn("forecast_generated_at", F.current_timestamp())

print("\n" + "═" * 80)
print("⏰ HOURLY CONGESTION PATTERN ANALYSIS")
print("═" * 80)
print(f"\n📊 Generated {hourly_forecast.count():,} hourly patterns (7 days × 24 hours)")
print("\n" + "─" * 80)
print("\n🔥 Top 10 Worst Congestion Periods:")
hourly_forecast.orderBy("congestion_score", ascending=False).select(
    "day_name",
    "hour_of_day",
    "total_hazards",
    "major_hazards",
    "congestion_score",
    "congestion_level"
).show(10, truncate=False)
print("═" * 80 + "\n")

In [0]:
# Analyze congestion by location
location_forecast = traffic_hazards.filter(
    F.col("location").isNotNull()
).groupBy("location").agg(
    # Hazard statistics
    F.count("*").alias("total_hazards"),
    F.sum(F.when(F.col("is_active"), 1).otherwise(0)).alias("active_hazards"),
    F.sum(F.when(F.col("severity") == "Major", 1).otherwise(0)).alias("major_hazards"),
    
    # Location data
    F.avg("latitude").alias("center_lat"),
    F.avg("longitude").alias("center_lon"),
    
    # Hazard types
    F.sum(F.when(F.col("main_category") == "Scheduled Roadwork", 1).otherwise(0)).alias("roadwork_count"),
    F.sum(F.when(F.col("main_category") == "Hazard", 1).otherwise(0)).alias("incident_count"),
    F.sum(F.when(F.col("main_category") == "Flooding", 1).otherwise(0)).alias("flood_count"),
    
    # Timing analysis
    F.min("start_time_ts").alias("earliest_hazard"),
    F.max("start_time_ts").alias("latest_hazard")
)

# Calculate risk score for each location (0-100)
location_forecast = location_forecast.withColumn(
    "risk_score",
    F.least(
        F.lit(100),
        (
            # Total hazard density (max 40 points)
            F.least(F.lit(40), F.col("total_hazards") / 2) +
            
            # Major hazard penalty (max 30 points)
            (F.col("major_hazards") * 10) +
            
            # Active hazard factor (max 30 points)
            F.least(F.lit(30), F.col("active_hazards") * 2)
        )
    ).cast("int")
)

# Categorize risk level
location_forecast = location_forecast.withColumn(
    "risk_level",
    F.when(F.col("risk_score") >= 70, "High Risk")
     .when(F.col("risk_score") >= 40, "Moderate Risk")
     .otherwise("Low Risk")
)

# Identify dominant hazard type
location_forecast = location_forecast.withColumn(
    "dominant_hazard_type",
    F.when(
        (F.col("roadwork_count") >= F.col("incident_count")) & 
        (F.col("roadwork_count") >= F.col("flood_count")),
        "Roadwork"
    ).when(
        (F.col("incident_count") >= F.col("roadwork_count")) & 
        (F.col("incident_count") >= F.col("flood_count")),
        "Incidents"
    ).when(
        F.col("flood_count") > 0,
        "Flooding"
    ).otherwise("Mixed")
)

location_forecast = location_forecast.withColumn("forecast_generated_at", F.current_timestamp())

print("\n" + "═" * 80)
print("📍 LOCATION-BASED CONGESTION FORECAST")
print("═" * 80)
print(f"\n📊 Analyzed {location_forecast.count():,} locations")
print("\n" + "─" * 80)
print("\n⚠️  Top 10 High-Risk Areas:")
location_forecast.orderBy("risk_score", ascending=False).select(
    "location",
    "total_hazards",
    "major_hazards",
    "active_hazards",
    "risk_score",
    "risk_level",
    "dominant_hazard_type"
).show(10, truncate=False)
print("═" * 80 + "\n")

In [0]:
# Generate optimal charging time recommendations based on congestion patterns

# Calculate average congestion by hour across all days
avg_hourly_congestion = hourly_forecast.groupBy("hour_of_day").agg(
    F.avg("congestion_score").alias("avg_congestion_score"),
    F.avg("total_hazards").alias("avg_hazards"),
    F.sum("total_hazards").alias("total_hazards_all_days"),
    F.max("congestion_score").alias("max_congestion_score")
)

# Create recommendations
recommendations = avg_hourly_congestion.withColumn(
    "recommendation_category",
    F.when(
        F.col("avg_congestion_score") < 20,
        "Excellent"
    ).when(
        F.col("avg_congestion_score") < 40,
        "Good"
    ).when(
        F.col("avg_congestion_score") < 60,
        "Fair"
    ).otherwise("Poor")
)

# Add time period labels
recommendations = recommendations.withColumn(
    "time_period",
    F.when(
        F.col("hour_of_day").between(0, 5),
        "Late Night"
    ).when(
        F.col("hour_of_day").between(6, 8),
        "Morning Rush"
    ).when(
        F.col("hour_of_day").between(9, 11),
        "Mid-Morning"
    ).when(
        F.col("hour_of_day").between(12, 14),
        "Midday"
    ).when(
        F.col("hour_of_day").between(15, 16),
        "Afternoon"
    ).when(
        F.col("hour_of_day").between(17, 19),
        "Evening Rush"
    ).when(
        F.col("hour_of_day").between(20, 22),
        "Evening"
    ).otherwise("Night")
)

# Add charging recommendation text
recommendations = recommendations.withColumn(
    "charging_recommendation",
    F.when(
        F.col("recommendation_category") == "Excellent",
        "Ideal time to charge - minimal traffic expected"
    ).when(
        F.col("recommendation_category") == "Good",
        "Good time to charge - low traffic conditions"
    ).when(
        F.col("recommendation_category") == "Fair",
        "Acceptable charging time - moderate traffic possible"
    ).otherwise(
        "Avoid if possible - high traffic expected"
    )
)

# Add travel recommendation
recommendations = recommendations.withColumn(
    "travel_recommendation",
    F.when(
        F.col("avg_congestion_score") >= 60,
        "Avoid travel if possible - plan alternate routes"
    ).when(
        F.col("avg_congestion_score") >= 40,
        "Allow extra travel time - monitor live traffic"
    ).when(
        F.col("avg_congestion_score") >= 20,
        "Normal travel conditions expected"
    ).otherwise(
        "Clear roads expected - optimal travel time"
    )
)

recommendations = recommendations.withColumn("generated_at", F.current_timestamp())

print("\n" + "═" * 80)
print("⚡ EV CHARGING TIME RECOMMENDATIONS")
print("═" * 80)
print(f"\n📊 Generated recommendations for {recommendations.count()} hours")
print("\n" + "─" * 80)
print("\n✅ Best Times to Charge (Top 5):")
recommendations.orderBy("avg_congestion_score").select(
    "hour_of_day",
    "time_period",
    F.round("avg_congestion_score", 1).alias("congestion"),
    "recommendation_category",
    "charging_recommendation"
).show(5, truncate=False)

print("\n" + "─" * 80)
print("\n❌ Worst Times to Travel (Top 5):")
recommendations.orderBy("avg_congestion_score", ascending=False).select(
    "hour_of_day",
    "time_period",
    F.round("avg_congestion_score", 1).alias("congestion"),
    "recommendation_category",
    "travel_recommendation"
).show(5, truncate=False)
print("═" * 80 + "\n")

In [0]:
# Compare weekday vs weekend congestion patterns
weekday_weekend = hourly_forecast.withColumn(
    "day_type",
    F.when(
        F.col("day_of_week").isin([1, 7]),  # Sunday=1, Saturday=7
        "Weekend"
    ).otherwise("Weekday")
)

# Aggregate by day type and hour
day_type_comparison = weekday_weekend.groupBy(
    "day_type",
    "hour_of_day"
).agg(
    F.avg("total_hazards").alias("avg_hazards"),
    F.avg("congestion_score").alias("avg_congestion"),
    F.sum("total_hazards").alias("total_hazards")
).orderBy("day_type", "hour_of_day")

print("\n" + "═" * 80)
print("📅 WEEKDAY vs WEEKEND CONGESTION PATTERNS")
print("═" * 80)

# Weekday peak hours
weekday_peak = day_type_comparison.filter(
    F.col("day_type") == "Weekday"
).orderBy("avg_congestion", ascending=False).first()

# Weekend peak hours
weekend_peak = day_type_comparison.filter(
    F.col("day_type") == "Weekend"
).orderBy("avg_congestion", ascending=False).first()

print(f"\n📊 Summary Statistics:")
print(f"\n   Weekday Peak:")
print(f"   • Hour: {weekday_peak['hour_of_day']}:00")
print(f"   • Avg Hazards: {weekday_peak['avg_hazards']:.1f}")
print(f"   • Congestion Score: {weekday_peak['avg_congestion']:.1f}/100")
print(f"\n   Weekend Peak:")
print(f"   • Hour: {weekend_peak['hour_of_day']}:00")
print(f"   • Avg Hazards: {weekend_peak['avg_hazards']:.1f}")
print(f"   • Congestion Score: {weekend_peak['avg_congestion']:.1f}/100")

print("\n" + "─" * 80)
print("\n📈 Hourly Comparison (Top 10 Hours):")
day_type_comparison.orderBy("avg_congestion", ascending=False).select(
    "day_type",
    "hour_of_day",
    F.round("avg_hazards", 1).alias("avg_hazards"),
    F.round("avg_congestion", 1).alias("congestion_score")
).show(10, truncate=False)
print("═" * 80 + "\n")

In [0]:
# Write all three forecast tables

print("\n" + "═" * 80)
print("💾 WRITING FORECAST TABLES")
print("═" * 80)

# 1. Hourly Forecast
hourly_count = hourly_forecast.count()
hourly_forecast.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(HOURLY_FORECAST_TABLE)
print(f"\n✅ Hourly Forecast:    {hourly_count:,} records → {HOURLY_FORECAST_TABLE}")

# 2. Location Forecast
location_count = location_forecast.count()
location_forecast.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(LOCATION_FORECAST_TABLE)
print(f"✅ Location Forecast:  {location_count:,} records → {LOCATION_FORECAST_TABLE}")

# 3. Recommendations
rec_count = recommendations.count()
recommendations.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(RECOMMENDATIONS_TABLE)
print(f"✅ Recommendations:    {rec_count:,} records → {RECOMMENDATIONS_TABLE}")

print("\n   📝 Write Mode:    OVERWRITE")
print("   📄 Format:        Delta Lake")
print("   🔄 Schema:        Auto-evolved")
print("═" * 80 + "\n")

In [0]:
print("\n" + "═" * 80)
print("🎯 CONGESTION FORECAST ENGINE - EXECUTIVE SUMMARY")
print("═" * 80)

# Load tables for validation
hourly_tbl = spark.table(HOURLY_FORECAST_TABLE)
location_tbl = spark.table(LOCATION_FORECAST_TABLE)
rec_tbl = spark.table(RECOMMENDATIONS_TABLE)

print("\n📊 Forecast Outputs:")
print(f"   • Hourly Patterns:     {hourly_tbl.count():,} records")
print(f"   • Location Analysis:   {location_tbl.count():,} LGA regions")
print(f"   • Recommendations:     {rec_tbl.count():,} hourly windows")

print("\n" + "─" * 80)
print("\n⏰ Peak Congestion Times:")
worst_times = hourly_tbl.orderBy("congestion_score", ascending=False).limit(3).collect()
for i, row in enumerate(worst_times, 1):
    print(f"   {i}. {row['day_name']} at {row['hour_of_day']}:00 - Score: {row['congestion_score']}/100 ({row['congestion_level']})")

print("\n" + "─" * 80)
print("\n✅ Best Charging Windows:")
best_times = rec_tbl.orderBy("avg_congestion_score").limit(3).collect()
for i, row in enumerate(best_times, 1):
    print(f"   {i}. Hour {row['hour_of_day']}:00 ({row['time_period']}) - Congestion: {row['avg_congestion_score']:.1f}/100")
    print(f"      → {row['charging_recommendation']}")

print("\n" + "─" * 80)
print("\n⚠️  High-Risk Locations:")
high_risk = location_tbl.filter(F.col("risk_level") == "High Risk").count()
moderate_risk = location_tbl.filter(F.col("risk_level") == "Moderate Risk").count()
print(f"   • High Risk LGAs:      {high_risk}")
print(f"   • Moderate Risk LGAs:  {moderate_risk}")
print(f"   • Total LGAs Analyzed: {location_tbl.count()}")

top_risk_area = location_tbl.orderBy("risk_score", ascending=False).first()
print(f"\n   Highest Risk: {top_risk_area['location']}")
print(f"   • Risk Score: {top_risk_area['risk_score']}/100")
print(f"   • Total Hazards: {top_risk_area['total_hazards']}")
print(f"   • Active Now: {top_risk_area['active_hazards']}")
print(f"   • Dominant Type: {top_risk_area['dominant_hazard_type']}")

print("\n" + "─" * 80)
print("\n🚗 Congestion Distribution:")
congestion_dist = hourly_tbl.groupBy("congestion_level").count().orderBy("congestion_level")
for row in congestion_dist.collect():
    pct = (row['count'] / hourly_tbl.count()) * 100
    print(f"   • {row['congestion_level']:12s}: {row['count']:3d} periods ({pct:5.1f}%)")

print("\n" + "─" * 80)
print("\n💡 Key Insights:")

# Calculate insights
total_periods = hourly_tbl.count()
critical_periods = hourly_tbl.filter(F.col("congestion_level") == "Critical").count()
peak_hour_hazards = hourly_tbl.filter(F.col("is_peak_hour")).agg(F.sum("total_hazards")).collect()[0][0]
off_peak_hazards = hourly_tbl.filter(~F.col("is_peak_hour")).agg(F.sum("total_hazards")).collect()[0][0]

print(f"   • Critical Congestion: {critical_periods}/{total_periods} time periods ({critical_periods/total_periods*100:.1f}%)")
print(f"   • Peak Hour Hazards: {peak_hour_hazards:,} ({peak_hour_hazards/(peak_hour_hazards+off_peak_hazards)*100:.1f}% of total)")
print(f"   • Off-Peak Hazards: {off_peak_hazards:,} ({off_peak_hazards/(peak_hour_hazards+off_peak_hazards)*100:.1f}% of total)")

excellent_hours = rec_tbl.filter(F.col("recommendation_category") == "Excellent").count()
print(f"   • Excellent Charging Windows: {excellent_hours}/24 hours available")

print("\n" + "─" * 80)
print("\n🎯 Recommendations for EV Drivers:")
print("   1. Charge during late night (12 AM - 5 AM) for minimal congestion")
print("   2. Avoid morning rush (7-9 AM) and evening rush (5-7 PM)")
print("   3. Plan routes around high-risk LGAs identified above")
print("   4. Check real-time traffic before departure")
print("   5. Consider mid-morning (10-11 AM) or early afternoon (2-4 PM) for travel")

print("\n" + "═" * 80)
print("✅ CONGESTION FORECAST ENGINE COMPLETE")
print("═" * 80 + "\n")